# TrialScope, step by step: from upsert to search

TrialScope searches a patient's notes, PubMed abstracts and ClinicalTrials.gov records in one place, on
[turbopuffer](https://turbopuffer.com). This notebook rebuilds its data layer from scratch with a small
dataset, one step at a time, so the app makes sense before you see it:

| Step | What you'll see |
| --- | --- |
| 1. The data | 20 trials, 20 PubMed abstracts, 20 synthetic patients and their 83 notes |
| 2. Schema design | One `text` attribute that is both keyword-indexed and embedded by turbopuffer |
| 3. Loading with upserts | Writing trials and abstracts, and what turbopuffer reports back |
| 4. One namespace per patient | The patient directory, and each patient's notes in its own namespace |
| 5. Editing a note | An upsert that replaces a row, readable immediately |
| 6. Lookups without search | Filtering and ordering by attributes |
| 7. Keyword search (BM25) | Exact terms, ranked by BM25 score |
| 8. Vector search | Meaning, with the query embedded by turbopuffer |
| 9. Hybrid search | Both lists in one request, fused with reciprocal rank fusion |
| 10. Filters | Trials this patient could actually join |
| 11. One search across sources | Notes, literature and trials together, as the app's search page does |
| 12. Where the copilot fits | How the AI copilot uses these same queries |

**Separate data.** Everything here lives in namespaces starting with `wt-` (walkthrough). The app uses `ts-`,
so nothing in this notebook reads or changes the app's data. The last cell deletes the `wt-` namespaces.

**Same data as the app.** The records were built with TrialScope's own loaders (`ingest/` in this repository):
the patients are the app's first 20 (same generator, same seed, Dana Whitfield first), and the trials and
abstracts come from ClinicalTrials.gov and PubMed through the loaders' `to_row` functions, chosen to match
those patients' conditions. `walkthrough/build_data.py` rebuilds them.

## Setup

You need a turbopuffer API key. In **GitHub Codespaces**, add it as a Codespaces secret named
`TURBOPUFFER_API_KEY` (GitHub settings → Codespaces → Secrets, with this repository selected) before
creating the codespace. In **Google Colab**, add it under the key icon (Secrets) with the same name.
Otherwise the next cell asks for it.

In [ ]:
import json, os, pathlib, time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
import turbopuffer

pd.set_option("display.max_colwidth", 90)


def api_key() -> str:
    if os.environ.get("TURBOPUFFER_API_KEY"):
        return os.environ["TURBOPUFFER_API_KEY"]
    try:  # Google Colab secrets
        from google.colab import userdata
        return userdata.get("TURBOPUFFER_API_KEY")
    except Exception:
        from getpass import getpass
        return getpass("turbopuffer API key: ")


tpuf = turbopuffer.Turbopuffer(api_key=api_key(), region=os.environ.get("TURBOPUFFER_REGION", "aws-us-east-1"))

PREFIX = "wt-"  # the app uses "ts-"; this notebook only ever touches "wt-"
NS_TRIALS, NS_PUBMED, NS_PATIENTS = PREFIX + "trials", PREFIX + "pubmed", PREFIX + "patients"
patient_ns = lambda pid: f"{PREFIX}patient-{pid}"


def ns(name: str):
    assert name.startswith(PREFIX), f"refusing to touch {name}: the walkthrough only uses {PREFIX}* namespaces"
    return tpuf.namespace(name)


print("Connected. Namespaces used here start with", repr(PREFIX))

In [ ]:
# Small helpers for showing results as tables.

def row_dict(row) -> dict:
    d = row.model_dump() if hasattr(row, "model_dump") else dict(row)
    d.update(getattr(row, "model_extra", None) or {})
    d.pop("vector", None)
    return d


def table(rows, cols, score=None):
    out = []
    for i, r in enumerate(rows, 1):
        d = row_dict(r)
        item = {"#": i}
        if score:
            item[score] = round(d.get("$dist", float("nan")), 4)
        item.update({c: d.get(c) for c in cols})
        out.append(item)
    return pd.DataFrame(out).set_index("#") if out else pd.DataFrame()


def perf(res) -> str:
    p = res.performance
    parts = [f"server {p.server_total_ms} ms"]
    if getattr(p, "embedding_ms", None):
        parts.append(f"embedding {p.embedding_ms} ms ({p.embedding_tokens} tokens)")
    if getattr(p, "approx_namespace_size", None) is not None:
        parts.append(f"~{p.approx_namespace_size} docs in namespace")
    return ", ".join(parts)

## 1. The data

Four kinds of record, already shaped the way the app stores them.

In [ ]:
DATA = pathlib.Path("data") if pathlib.Path("data").exists() else pathlib.Path("walkthrough/data")
RAW = "https://raw.githubusercontent.com/jgerson/trialscope-demo/main/walkthrough/data/"


def load(name):
    path = DATA / f"{name}.json"
    if path.exists():
        return json.loads(path.read_text())
    import urllib.request  # Colab without the repository checked out
    return json.loads(urllib.request.urlopen(RAW + f"{name}.json").read())


trials, pubmed, patients, notes = (load(n) for n in ("trials", "pubmed", "patients", "notes"))
print(f"{len(trials)} trials, {len(pubmed)} abstracts, {len(patients)} patients, {len(notes)} notes")
pd.DataFrame(patients)[["id", "name", "age", "sex", "city", "primary_dx", "stage", "biomarkers", "note_count"]]

Each record has a `text` field built for searching, plus fields for display and for filtering. A trial:

In [ ]:
t = trials[0]
print(t["text"][:700], "…\n")
{k: v for k, v in t.items() if k != "text"}

## 2. Schema design

The key decision: **one `text` attribute does double duty.** It is indexed for keyword search (BM25)
*and* turbopuffer embeds it itself with `voyage-4-lite`, so the app never runs an embedding model or stores
vectors of its own. Writes send plain text; turbopuffer computes and stores the vector.

The other attributes are either **stored for display** (`filterable: False`, which saves index space) or
**filterable metadata** with a type, used for exact filters like status, age range and sex. These are the
app's schemas, copied from `tp_client.py`.

In [ ]:
EMBED_MODEL = "voyage/voyage-4-lite"
TEXT = {"type": "string", "full_text_search": True, "embed": EMBED_MODEL}  # BM25 + native embedding
STORED = {"type": "string", "filterable": False}                           # shown, never filtered

SCHEMAS = {
    "trials": {"text": TEXT, "title": STORED, "summary": STORED, "nct_id": {"type": "string"},
               "status": {"type": "string"}, "phases": {"type": "[]string"}, "conditions": {"type": "[]string"},
               "countries": {"type": "[]string"}, "sponsor": {"type": "string"}, "start_year": {"type": "int"},
               "min_age": {"type": "int"}, "max_age": {"type": "int"}, "sex": {"type": "string"}},
    "pubmed": {"text": TEXT, "title": STORED, "abstract": STORED, "pmid": {"type": "string"},
               "journal": {"type": "string"}, "year": {"type": "int"}, "pub_types": {"type": "[]string"},
               "mesh": {"type": "[]string"}, "topic": {"type": "string"}},
    "notes": {"text": TEXT, "title": STORED, "body": STORED, "note_type": {"type": "string"},
              "visit_date": {"type": "string"}, "author": {"type": "string"}},
}
pd.DataFrame([{"attribute": k, "type": v["type"],
               "keyword index": "yes" if v.get("full_text_search") else "",
               "embedded": v.get("embed", ""), "filterable": "no" if v.get("filterable") is False else "yes"}
              for k, v in SCHEMAS["trials"].items()])

## 3. Loading with upserts

A write with `upsert_rows` inserts rows whose `id` is new and replaces rows whose `id` already exists, so
loading is safe to repeat. The schema travels with the write; turbopuffer creates the namespace on the
first one. `distance_metric` says how vectors are compared.

The response is worth reading: rows upserted, billable bytes, and how long the server spent, including
embedding.

In [ ]:
def upsert(name, rows, kind=None):
    kwargs = {"upsert_rows": rows}
    if kind:
        kwargs.update(schema=SCHEMAS[kind], distance_metric="cosine_distance")
    t0 = time.perf_counter()
    res = ns(name).write(**kwargs)
    took = (time.perf_counter() - t0) * 1000
    p = res.performance
    return {"namespace": name, "rows upserted": res.rows_upserted, "billable bytes": res.billing.billable_logical_bytes_written,
            "round trip ms": round(took), "server ms": getattr(p, "server_total_ms", None),
            "embedding ms": getattr(p, "embedding_ms", None), "tokens embedded": getattr(p, "embedding_tokens", None)}


pd.DataFrame([upsert(NS_TRIALS, trials, "trials"), upsert(NS_PUBMED, pubmed, "pubmed")])

Most of the server time is embedding: turbopuffer turned each record's `text` into a vector during the write.
Run the cell again and the counts stay the same, because the ids already exist and the rows are replaced.

## 4. One namespace per patient

The **patient directory** (`wt-patients`) is a plain table: no `text`, no vectors, just attributes to list
and sort. Each patient's **notes** go in their own namespace, `wt-patient-<id>`. That is multi-tenancy by
namespace: a search can only ever see one patient's notes, and removing a patient is deleting one namespace.
turbopuffer is built for very many small namespaces, so 20 here (or millions) is fine.

In [ ]:
directory = upsert(NS_PATIENTS, patients)  # no schema: attributes only

by_patient = {}
for n in notes:
    row = {k: v for k, v in n.items() if k != "patient_id"}
    by_patient.setdefault(n["patient_id"], []).append(row)

with ThreadPoolExecutor(max_workers=8) as pool:
    results = list(pool.map(lambda pid: upsert(patient_ns(pid), by_patient[pid], "notes"), by_patient))

print(f"Directory: {directory['rows upserted']} patients in {NS_PATIENTS}")
pd.DataFrame(results)[["namespace", "rows upserted", "server ms", "embedding ms"]].head(8)

## 5. Editing a note: the upsert again

This is what happens when you save a note in the app. First check whether the id exists (that decides
"insert" or "update"), then write. The query right after returns the new text: turbopuffer queries are
**strongly consistent** by default, so a write is visible as soon as it's acknowledged.

In [ ]:
DANA = "0001"
dana_ns = ns(patient_ns(DANA))
latest = max(by_patient[DANA], key=lambda n: n["visit_date"])
edited = {**latest, "body": latest["body"] + "\n\nPatient reports new grade 1 tingling in the fingertips since cycle 4."}
edited["text"] = f"{edited['title']}\n{edited['body']}"

before = dana_ns.query(filters=("id", "Eq", edited["id"]), top_k=1, include_attributes=["title"])
print("Exists before the write:", bool(before.rows), "→", "update" if before.rows else "insert")

t0 = time.perf_counter()
w = dana_ns.write(upsert_rows=[edited], schema=SCHEMAS["notes"], distance_metric="cosine_distance")
t1 = time.perf_counter()
found = dana_ns.query(rank_by=("text", "BM25", "tingling fingertips"), top_k=3, include_attributes=["title", "visit_date"])
t2 = time.perf_counter()

print(f"Write acknowledged in {(t1 - t0) * 1000:.0f} ms; rows upserted: {w.rows_upserted}")
print(f"Search for the new words {(t2 - t1) * 1000:.0f} ms later finds:")
table(found.rows, ["title", "visit_date"], score="BM25 score")

## 6. Lookups without search

Not every read is a search. The app lists patients by name and a patient's notes newest first by ordering
on an attribute, and narrows trials with filters. No ranking model is involved.

In [ ]:
notes_res = dana_ns.query(rank_by=("visit_date", "desc"), top_k=20, include_attributes=["title", "note_type", "visit_date"])
print("Dana's notes, newest first:")
display(table(notes_res.rows, ["visit_date", "note_type", "title"]))

recruiting = ns(NS_TRIALS).query(filters=("status", "In", ["Recruiting", "Not yet recruiting"]),
                                 top_k=50, include_attributes=["nct_id", "status"])
print(f"{len(recruiting.rows)} of {len(trials)} trials are recruiting or about to")

## 7. Keyword search (BM25)

BM25 ranks by the query's words, weighting rare words more. It is exact: great for drug names, gene names
and trial ids, blind to synonyms. Higher score is better.

In [ ]:
q = "trastuzumab pertuzumab HER2"
res = ns(NS_TRIALS).query(rank_by=("text", "BM25", q), top_k=5, include_attributes=["nct_id", "status", "title"])
print(f"BM25 for {q!r}: {perf(res)}")
table(res.rows, ["nct_id", "status", "title"], score="BM25 score")

## 8. Vector search

Now a question in plain words that shares almost no vocabulary with the abstracts. `("Embed", query)` asks
turbopuffer to embed the query with the same model as the documents, then find the nearest vectors. The
score is cosine distance: **lower is closer**. Compare with BM25 on the same words.

In [ ]:
q = "heart damage caused by breast cancer drugs"
vec = ns(NS_PUBMED).query(rank_by=("text", "ANN", ("Embed", q)), top_k=5, include_attributes=["title", "year"])
kw = ns(NS_PUBMED).query(rank_by=("text", "BM25", q), top_k=5, include_attributes=["title", "year"])
print(f"Vector: {perf(vec)}")
display(table(vec.rows, ["year", "title"], score="cosine distance"))
print("BM25 on the same words:")
table(kw.rows, ["year", "title"], score="BM25 score")

Vector search finds the cardiotoxicity papers even though they say "cardiotoxicity" and "trastuzumab", not
"heart damage" and "breast cancer drugs". BM25 can only match the words it was given.

## 9. Hybrid search

The app's default sends **both queries in one request** (`multi_query`) and fuses the two ranked lists with
**reciprocal rank fusion**: each document scores the sum of `1 / (60 + rank)` over the lists it appears in.
Appearing in both lists beats a high rank in one. The app fuses in its own code, rather than asking
turbopuffer to, so it can show each document's rank in each list.

In [ ]:
RRF_K = 60


def hybrid(name, q, filters=None, top_k=8, attrs=("title",)):
    base = {"top_k": top_k, "include_attributes": list(attrs)}
    if filters:
        base["filters"] = filters
    res = ns(name).multi_query(queries=[{**base, "rank_by": ("text", "ANN", ("Embed", q))},
                                        {**base, "rank_by": ("text", "BM25", q)}])
    docs = {}
    for list_name, result in zip(("vector", "bm25"), res.results):
        for rank, row in enumerate(result.rows or [], 1):
            d = row_dict(row)
            doc = docs.setdefault(d["id"], {**{a: d.get(a) for a in attrs}, "vector rank": None, "bm25 rank": None})
            doc[f"{list_name} rank"] = rank
    for doc in docs.values():
        doc["fused"] = round(sum(1 / (RRF_K + r) for r in (doc["vector rank"], doc["bm25 rank"]) if r), 5)
    ordered = sorted(docs.items(), key=lambda kv: kv[1]["fused"], reverse=True)[:top_k]
    return pd.DataFrame([{"id": k, **v} for k, v in ordered], index=range(1, len(ordered) + 1)), res


df, res = hybrid(NS_PUBMED, "trastuzumab cardiotoxicity monitoring", attrs=("title", "year"))
print(f"One multi_query request: {perf(res)}")
df

## 10. Filters: trials this patient could join

Filters are exact and run inside the search. The app's "match the patient" option adds age and sex
conditions from the patient record, and the copilot usually asks for recruiting trials only. Here is
Dana (58, female) against the trials, with and without those filters.

In [ ]:
dana = next(p for p in patients if p["id"] == DANA)
match = ("And", [
    ("status", "In", ["Recruiting", "Not yet recruiting"]),
    ("min_age", "Lte", dana["age"]),
    ("max_age", "Gte", dana["age"]),
    ("sex", "In", [dana["sex"], "All"]),
])
q = "HER2-positive breast cancer after neoadjuvant therapy"
attrs = ("nct_id", "status", "min_age", "max_age", "sex", "title")
unfiltered, _ = hybrid(NS_TRIALS, q, attrs=attrs, top_k=6)
filtered, _ = hybrid(NS_TRIALS, q, filters=match, attrs=attrs, top_k=6)
print("Without filters:")
display(unfiltered[["nct_id", "status", "min_age", "max_age", "sex", "title", "fused"]])
print("Recruiting, and open to a 58-year-old woman:")
display(filtered[["nct_id", "status", "min_age", "max_age", "sex", "title", "fused"]])

# Which trials can the filter never return for Dana, and why? (A filter-only query, no ranking.)
allowed = {row_dict(r)["id"] for r in ns(NS_TRIALS).query(filters=match, top_k=100, include_attributes=["nct_id"]).rows}


def why_not(t):
    reasons = []
    if t["status"] not in ("Recruiting", "Not yet recruiting"):
        reasons.append(f"status is {t['status']}")
    if not t["min_age"] <= dana["age"] <= t["max_age"]:
        reasons.append(f"ages {t['min_age']} to {t['max_age']}")
    if t["sex"] not in (dana["sex"], "All"):
        reasons.append(f"sex is {t['sex']}")
    return ", ".join(reasons)


excluded = [{"nct_id": t["nct_id"], "excluded because": why_not(t), "title": t["title"]} for t in trials if t["id"] not in allowed]
print(f"Filtered out for Dana: {len(excluded)} of {len(trials)} trials")
pd.DataFrame(excluded)

The last table is the point: those trials can never appear in Dana's results, however well they match the
words, because the filter runs before ranking. The ranking only ever compares trials she could actually join.

## 11. One search across sources

The app's search page and its copilot search several namespaces at once: this patient's notes, the
literature and the trials. Each namespace runs its own hybrid search **in parallel**, then the lists are
merged by the same rank-fusion idea, so a note, an abstract and a trial can be compared fairly even though
their scores aren't.

In [ ]:
def search_all(q, patient_id, top_k=4):
    targets = [(patient_ns(patient_id), "note", None), (NS_PUBMED, "pubmed", None), (NS_TRIALS, "trial", None)]
    t0 = time.perf_counter()
    with ThreadPoolExecutor(max_workers=3) as pool:
        parts = list(pool.map(lambda t: (t[1], hybrid(t[0], q, t[2], top_k)[0]), targets))
    wall = (time.perf_counter() - t0) * 1000
    merged = []
    for source, df in parts:
        for rank, row in enumerate(df.itertuples(), 1):
            merged.append({"source": source, "rank in source": rank, "merged score": round(1 / (RRF_K + rank), 5),
                           "title": row.title})
    merged.sort(key=lambda m: m["merged score"], reverse=True)
    print(f"3 namespaces searched in parallel in {wall:.0f} ms")
    return pd.DataFrame(merged, index=range(1, len(merged) + 1))


search_all("cardiac monitoring during trastuzumab", DANA)

## 12. Where the copilot fits

The app's copilot is an AI model with two tools, and both are the queries above:

- **`read_patient_notes`** is step 6: Dana's notes, newest first.
- **`search`** is steps 9 to 11: hybrid search over the sources it picks, with the filters it picks
  (recruiting only, matching the patient's age and sex).

The model reads the notes, decides what to search for, and writes an answer that cites each result by tag
(`[N1]`, `[T2]`). The app then checks each claim against the cited text. Nothing about the data layer
changes for the AI: it uses the same namespaces, schemas and queries you just ran.

## From this notebook to the app

| In this notebook | In the app |
| --- | --- |
| Steps 3 and 4: upserts with native embedding | How the data was loaded (`ingest/` scripts) |
| Step 5: an upsert, readable immediately | Saving a note, and the save panel's timing breakdown |
| Step 6: ordered lookups | The patient list and the notes list |
| Steps 7 to 9: BM25, vector, hybrid | The search page's mode switch and its **Explain** panel |
| Step 10: filters | Trial filters and "match the patient" |
| Step 11: several namespaces, merged | The search page's "All" scope, and the copilot's searches |
| Step 12 | The copilot, with its tool calls shown step by step |

Now open the app and run the same searches.

## Clean up

Deletes every `wt-` namespace this notebook created. The app's `ts-` data is never touched: the code
refuses any name without the `wt-` prefix. Set `CLEAN_UP = True` and run the cell.

In [ ]:
CLEAN_UP = False

names = [n.id for n in tpuf.namespaces(prefix=PREFIX)]
print(f"{len(names)} walkthrough namespaces:", ", ".join(names[:6]) + (" …" if len(names) > 6 else ""))
if CLEAN_UP:
    for name in names:
        ns(name).delete_all()
    print("Deleted.")